# Whisper large-v3 benchmark: Avaliação de Dialetos Regionais e Métricas de Precisão

## Objetivos e Metodologia
Sistemas de Reconhecimento Automático de Fala (ASR) como o OpenAI Whisper possuem alta acurácia geral, mas podem apresentar variações em sotaques regionais do português brasileiro.

Este notebook executa um pipeline de auditoria para quantificar:
- **Word Error Rate (WER):** Taxa de erro de palavras calculada comparando a transcrição do modelo com a transcrição humana de referência (ground truth) via `jiwer`.
- **Real-Time Factor (RTF):** Eficiência computacional medida pela razão entre o tempo de inferência e a duração do áudio.

---
> ⚙️ **Configuração de Hardware:** Antes de iniciar, acesse o menu *Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU*.

# 1. Configuração do ambiente

## 1.1 Instalação de Dependências e Verificação de GPU

In [ ]:
!pip install -q openai-whisper jiwer
import torch
print('GPU disponível:', torch.cuda.is_available())
!pip show openai-whisper | grep Version

#2. Ingestão e Processamento dos Áudios
 Utilize **apenas uma** das opções abaixo para disponibilizar as amostras de áudio no ambiente antes da transcrição:
> - **2.1:** Automação de extração direta via `yt-dlp`.
> - **2.2:** Upload manual do arquivo compactado `audios.zip`.

##2.1 Coleta e Automação via yt-dlp (recomendado)


In [ ]:
#Gera a estrutura de metadados (`clips.csv`) e executa o download automatizado dos trechos de 80 segundos diretamente da fonte, padronizando o formato em 16 kHz mono.

import pandas as pd

#Definição das URLs, tempos de início e duração dos cortes dos dialetos
dados_clips = [
    {"id": "F1", "url": "https://www.youtube.com/watch?v=EXEMPLO1", "inicio": "01:15", "duracao_s": 80},
    {"id": "F2", "url": "https://www.youtube.com/watch?v=EXEMPLO2", "inicio": "03:40", "duracao_s": 80},
    {"id": "F3", "url": "https://www.youtube.com/watch?v=EXEMPLO3", "inicio": "00:50", "duracao_s": 80}
]

df = pd.DataFrame(dados_clips)
df.to_csv("clips.csv", index=False, encoding="utf-8")
print("✅ Ficheiro clips.csv gerado com sucesso!")

In [ ]:
import csv, subprocess, sys
from pathlib import Path

# Pasta de saída para os áudios cortados
OUT = Path("audios")
OUT.mkdir(exist_ok=True)

def para_seg(t):
    """Converte o tempo de formato 'MM:SS' ou 'HH:MM:SS' para segundos totais."""
    p = [int(x) for x in t.split(":")]
    s = 0
    for x in p:
        s = s * 60 + x
    return s

def hms(s):
    """Converte segundos para o formato 'H:MM:SS' exigido pelo yt-dlp."""
    return f"{s//3600}:{(s%3600)//60:02d}:{s%60:02d}"

def duracao(arq):
    """Obtém a duração real do áudio extraído usando o ffprobe."""
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "default=nw=1:nk=1", str(arq)], capture_output=True, text=True)
    return float(r.stdout.strip())

# Loop principal: lê o clips.csv, baixa os trechos e faz o corte de 80s
for row in csv.DictReader(open("clips.csv", encoding="utf-8")):
    fid, url = row["id"], row["url"].strip()

    if not url or not row["inicio"].strip():
        print(f"[pulado] {fid}: sem link ou início")
        continue

    alvo = OUT / f"{fid}.wav"
    if alvo.exists():
        print(f"[ok] {fid} já existe")
        continue

    ini = para_seg(row["inicio"])
    dur = int(row["duracao_s"])

    # Comando yt-dlp otimizado para baixar apenas a seção de tempo necessária e converter para 16kHz mono
    cmd = ["yt-dlp", "-x", "--audio-format", "wav",
           "--download-sections", f"*{hms(ini)}-{hms(ini+dur)}",
           "--force-keyframes-at-cuts",
           "--postprocessor-args", "ffmpeg:-ar 16000 -ac 1",
           "-o", str(OUT / f"{fid}.%(ext)s"), url]

    print(f"[baixando] {fid} {row['inicio']} +{dur}s")
    if subprocess.run(cmd).returncode != 0:
        print(f"[ERRO] {fid}: falhou (veja a mensagem acima)")
        continue

    try:
        d = duracao(alvo)
        aviso = "" if abs(d - dur) <= 1 else "  <-- CONFERIR: duração diferente de 80 s"
        print(f"[feito] {fid}: {d:.1f} s{aviso}")
    except Exception as e:
        print(f"[feito] {fid} (não consegui conferir a duração: {e})")

## 2.2 Upload de Arquivo Zip

In [ ]:
#Realiza a extração local do ficheiro compactado contendo as amostras de áudio previamente cortadas.
from google.colab import files
import zipfile

enviado = files.upload()  # selecione audios.zip
zipfile.ZipFile('audios.zip.zip').extractall('.')
!ls audios

## 2.3 Transcrição via Faster-Whisper e cálculo RTF

In [ ]:
"""Etapa 1: transcreve todos os .wav da pasta 'audios' com o Whisper large-v3
e mede o tempo de processamento (3 execucoes por audio, usa a mediana).

Saidas (pasta 'saida'):
  - F1.txt, F3.txt, ...  -> transcricao do Whisper de cada fragmento
  - tempos.csv           -> duracao, tempos, mediana e fator de tempo real (RTF)
"""
import csv
import statistics
import time
from pathlib import Path

import torch
import whisper

PASTA_AUDIOS = Path("audios")
PASTA_SAIDA = Path("saida")
MODELO = "large-v3"
EXECUCOES = 3

PASTA_SAIDA.mkdir(exist_ok=True)
arquivos = sorted(PASTA_AUDIOS.glob("*.wav"), key=lambda p: int(p.stem[1:]))
if not arquivos:
    raise SystemExit("Nenhum .wav encontrado na pasta 'audios'.")

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {dispositivo} | Modelo: {MODELO} | Whisper: {getattr(whisper, '__version__', '?')}")
if dispositivo == "cpu":
    print("AVISO: sem GPU. O large-v3 vai ser MUITO lento; no Colab, ative a GPU T4.")

modelo = whisper.load_model(MODELO, device=dispositivo)  # carregamento fora da medicao

OPCOES = dict(language="pt", task="transcribe", temperature=0.0, fp16=(dispositivo == "cuda"))


def transcrever(caminho):
    inicio = time.perf_counter()
    resultado = modelo.transcribe(str(caminho), **OPCOES)
    return resultado["text"].strip(), time.perf_counter() - inicio


# execucao de aquecimento (nao entra na medicao)
print("Aquecimento...")
transcrever(arquivos[0])

linhas = []
for arq in arquivos:
    duracao = len(whisper.load_audio(str(arq))) / whisper.audio.SAMPLE_RATE
    tempos, texto = [], ""
    for i in range(EXECUCOES):
        txt, dt = transcrever(arq)
        tempos.append(dt)
        if i == 0:
            texto = txt
    (PASTA_SAIDA / f"{arq.stem}.txt").write_text(texto, encoding="utf-8")
    mediana = statistics.median(tempos)
    linhas.append({
        "fragmento": arq.stem,
        "duracao_s": round(duracao, 2),
        **{f"tempo_{i+1}_s": round(t, 2) for i, t in enumerate(tempos)},
        "mediana_s": round(mediana, 2),
        "rtf": round(mediana / duracao, 3),
    })
    print(f"{arq.stem}: mediana {mediana:.1f}s | RTF {mediana/duracao:.2f}")

with open(PASTA_SAIDA / "tempos.csv", "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=list(linhas[0].keys()))
    escritor.writeheader()
    escritor.writerows(linhas)
print("Pronto. Veja a pasta 'saida'.")


# 3. Ingestão de Ground Truth (Transcrições Humanas de Referência)

##3.1 Importação e Descompactação do Arquivo de Referências (`refs.zip`)

In [ ]:
from google.colab import files
import zipfile

enviado = files.upload()  # selecione referencias.zip
zipfile.ZipFile('refs.zip').extractall('referencias')
!ls referencias

# 4. Cálculo do Word Error Rate (WER) e Mapeamento de Erros

Calculo da acurácia do modelo comparando a transcrição gerada pelo Whisper com a **Ground Truth humana** através do alinhamento de distância de Levenshtein (via biblioteca `jiwer`).


**Metodologia de Limpeza**<br>                                                 Para garantir que a medição seja estritamente focada em acertos e erros de palavras — e não em diferenças de formatação —, a função de normalização é aplicada diretamente em memória durante a leitura dos arquivos:
- Conversão para letras minúsculas (`lower()`).
- Remoção de pontuações e símbolos especiais.
- Eliminação de quebras de linha e múltiplos espaços.

O processamento gera o arquivo `saida/wer_detalhado.csv` contendo as métricas de **Substituições (S)**, **Deleções/Remoções (D)**, **Inserções (I)** e acertos (**Hits**) para cada amostra regional.

In [ ]:
"""
Etapa 2: Processamento e cálculo das métricas de erro de transcrição (WER).
Entradas:
  - referencias/{frag}.txt -> Transcrição humana de referência
  - saida/{frag}.txt       -> Transcrição de hipótese (Whisper)
Saída:
  - saida/wer_detalhado.csv -> Mapeamento de erros por fragmento
"""
import csv
import re
from pathlib import Path
import jiwer

PASTA_REF = Path("referencias")
PASTA_SAIDA = Path("saida")


def normalizar(texto: str) -> str:
    """Aplica minúsculas, remove pontuações e normaliza espaços mantendo acentos."""
    texto = texto.lower()
    texto = re.sub(r"[^\w\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


def calcular_wer_fragmento(path_ref: Path, path_hip: Path) -> dict:
    """Calcula métricas detalhadas de WER para um único par referência/hipótese."""
    ref = normalizar(path_ref.read_text(encoding="utf-8"))
    hip = normalizar(path_hip.read_text(encoding="utf-8"))

    r = jiwer.process_words(ref, hip)

    return {
        "fragmento": path_ref.stem,
        "palavras_referencia": len(ref.split()),
        "palavras_hipotese": len(hip.split()),
        "wer": round(r.wer, 4),
        "substituicoes": r.substitutions,
        "remocoes": r.deletions,
        "insercoes": r.insertions,
        "hits": r.hits,
    }


def executar_calculo_wer() -> list[dict]:
    PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    resultados = []

    for ref_path in sorted(PASTA_REF.glob("*.txt"), key=lambda p: int(p.stem[1:])):
        frag = ref_path.stem
        hip_path = PASTA_SAIDA / f"{frag}.txt"

        if not hip_path.exists():
            print(f"⚠️ {frag}: sem transcrição em '{hip_path}', pulando.")
            continue

        m = calcular_wer_fragmento(ref_path, hip_path)
        resultados.append(m)
        print(f"✓ {frag}: WER {m['wer']:.1%} (S:{m['substituicoes']}, D:{m['remocoes']}, I:{m['insercoes']})")

    if not resultados:
        raise SystemExit("Erro: Nenhum par de transcrição foi encontrado.")

    # Exporta resultados do WER
    out_csv = PASTA_SAIDA / "wer_detalhado.csv"
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(resultados[0].keys()))
        writer.writeheader()
        writer.writerows(resultados)

    print(f"\n✅ Bloco WER concluído: '{out_csv}' gerado.")
    return resultados


if __name__ == "__main__":
    executar_calculo_wer()

# 5. Consolidação do Dataset para Análise / Dashboard
Cruzamento dos metadados sociolinguísticos dos falantes (região, dialeto e cidade) com as métricas de **WER** e **RTF** obtidas.

O resultado é exportado no arquivo `base_dashboard.csv`, pronto para alimentação de relatórios e visualizações analíticas.

In [ ]:
"""
Etapa 3: Cruzamento de métricas de WER, tempos de inferência e metadados.
Entradas:
  - saida/wer_detalhado.csv -> Métricas geradas no bloco de WER
  - saida/tempos.csv        -> Tempos de inferência e RTF
Saída:
  - saida/base_dashboard.csv -> Dataset consolidado para o dashboard
"""
import csv
from pathlib import Path

PASTA_SAIDA = Path("saida")

INFO = {
    "F1": ("Paulistano urbano (controle)", "A", "São Paulo (SP)"),
    "F3": ("Amazônico", "A", "Barcarena (PA)"),
    "F4": ("Amazônico", "B", "Manaus (AM)"),
    "F5": ("Nordestino", "A", "Recife (PE)"),
    "F6": ("Nordestino", "B", "Fortaleza (CE)"),
    "F7": ("Baiano", "A", "Salvador (BA)"),
    "F8": ("Baiano", "B", ""),
    "F9": ("Fluminense", "A", "Rio de Janeiro (RJ)"),
    "F10": ("Fluminense", "B", "Vitória (ES)"),
    "F11": ("Mineiro", "A", "Belo Horizonte (MG)"),
    "F12": ("Mineiro", "B", ""),
    "F13": ("Sulista", "A", "Cornélio Procópio (PR)"),
    "F14": ("Sulista", "B", "Florianópolis (SC)"),
}


def carregar_csv_map(caminho_csv: Path, chave: str) -> dict:
    """Carrega um CSV e indexa por uma chave específica."""
    dados = {}
    if caminho_csv.exists():
        with open(caminho_csv, encoding="utf-8") as f:
            for row in csv.DictReader(f):
                dados[row[chave]] = row
    return dados


def construir_base_dashboard():
    path_wer = PASTA_SAIDA / "wer_detalhado.csv"
    path_tempos = PASTA_SAIDA / "tempos.csv"

    if not path_wer.exists():
        raise FileNotFoundError("Execute 'etapa2_wer.py' primeiro para gerar 'wer_detalhado.csv'.")

    map_wer = carregar_csv_map(path_wer, "fragmento")
    map_tempos = carregar_csv_map(path_tempos, "fragmento")

    base_consolidada = []

    for frag, metadados in INFO.items():
        dialeto, falante, cidade = metadados
        wer_data = map_wer.get(frag)

        if not wer_data:
            continue  # Pula fragmentos que não possuem processamento de WER

        tempo_data = map_tempos.get(frag, {})

        linha_dashboard = {
            "fragmento": frag,
            "dialeto": dialeto,
            "falante": falante,
            "cidade": cidade,
            "palavras_referencia": wer_data.get("palavras_referencia", 0),
            "palavras_hipotese": wer_data.get("palavras_hipotese", 0),
            "wer": wer_data.get("wer", ""),
            "substituicoes": wer_data.get("substituicoes", 0),
            "remocoes": wer_data.get("remocoes", 0),
            "insercoes": wer_data.get("insercoes", 0),
            "hits": wer_data.get("hits", 0),
            "mediana_s": tempo_data.get("mediana_s", ""),
            "rtf": tempo_data.get("rtf", ""),
        }
        base_consolidada.append(linha_dashboard)

    out_dashboard = PASTA_SAIDA / "base_dashboard.csv"
    with open(out_dashboard, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(base_consolidada[0].keys()))
        writer.writeheader()
        writer.writerows(base_consolidada)

    print(f"✅ Base do dashboard criada com sucesso: '{out_dashboard}'.")


if __name__ == "__main__":
    construir_base_dashboard()